# Au [100] Verification: Multislice vs WPM vs Wave ODE

Thickness sweep for Au [100] comparing Fresnel, angular spectrum, WPM, and the second-order wave ODE. Publication plots use the full KG ODE result as the internal reference curve.

**Methods**: Fresnel (Paraxial), Angular Spectrum (Non-Paraxial), WPM, Second-Order Wave ODE

**Parameters**: Au FCC, a = 4.08 Å, 300 keV, 128 slices/cell with configurable x/y real-space pixel size, Weickenmeier-Kohl parametrization for the ODE reference, Lobato for Fresnel/Angular Spectrum/WPM.

In [ ]:
%matplotlib widget

import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".1"

import abtem
import cupy
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from ase.build import bulk
from scipy.special import erfc as scipy_erfc
from tqdm.auto import tqdm

from wide_angle_propagation.notebook_utils import (
    beam_amplitude_normalized,
    beam_for_angle_mrad,
    curve_rmse,
    global_phase_removed,
    grid_from_pixel_size,
    objective_lens_transfer_function,
    resolve_paper_figures_dir,
    shared_percentile_limits,
)
from wide_angle_propagation.propagation_methods import (
    angular_spectrum_propagation_kernel,
    energy2wavelength,
    fresnel_propagation_kernel,
    simulate_fresnel_as,
    simulate_kg_ode_full,
    simulate_wpm,
)

abtem.config.set({"device": "gpu"})
abtem.config.set({"precision": "float64"})
jax.config.update("jax_enable_x64", True)


## Crystal setup and Weickenmeier-Kohl parametrization

Au FCC, a = 4.076 Å, 128 slices/cell, no thermal motion.
WK parametrization for forward KG methods to match paper.

In [ ]:
from abtem.parametrizations import Parametrization

def weickenmeier_kohl_function(k2, parameters):
    A, B = parameters
    s2 = k2 / 4.0
    s2_expanded = s2[..., None]
    term = -np.expm1(-B * s2_expanded)
    sum_term = np.sum(A * term, axis=-1)
    with np.errstate(divide='ignore', invalid='ignore'):
        f_s = sum_term / s2
    limit_val = np.sum(A * B)
    f_s = np.where(s2 == 0, limit_val, f_s)
    return f_s * 47.87801

def weickenmeier_kohl_potential(r, parameters):
    A, B = parameters
    r = np.asarray(r, dtype=np.float64)
    result = np.zeros_like(r)
    for i in range(len(A)):
        result += A[i] * scipy_erfc(2 * np.pi * r / np.sqrt(B[i]))
    with np.errstate(divide='ignore', invalid='ignore'):
        V = 47.87801 * 4 * np.pi * result / r
    V = np.where(r < 1e-14, 1e30, V)
    return V

class WeickenmeierKohlParametrization(Parametrization):
    def __init__(self):
        super().__init__(parameters={})
        self._functions = {
            'elastic': weickenmeier_kohl_function,
            'projected_scattering_factor': weickenmeier_kohl_function,
            'potential': weickenmeier_kohl_potential,
        }

    def scaled_parameters(self, symbol, name):
        if "Au" not in symbol:
            raise NotImplementedError("Only Au is implemented.")
        Z = 79
        V = 0.4
        B = np.array([5.493e-01, 1.728e+00, 6.720e+00, 2.637e-02, 7.253e-02, 3.546e+01])
        factor = 0.02395 * Z
        a1_val = factor / (3 * (1 + V))
        A = np.array([a1_val, a1_val, a1_val, V * a1_val, V * a1_val, V * a1_val])
        return [A, B]

    def cutoff(self, symbol):
        return 20.0

# --- Crystal parameters ---
a_central = 4.076
n_slices_per_cell = 128
energy = 300e3
n_cells_range = range(0, 101)

# Change these two values to use rectangular real-space pixels. The integer grid
# is chosen as the nearest grid compatible with the Au unit-cell extent.
pixel_size_x = 0.0318  # Angstrom
pixel_size_y = 0.0318  # Angstrom

beam_target_mrad = 135.0
beam_target_axis = "y"

atoms = bulk("Au", "fcc", a=a_central, cubic=True)
atoms.info['thermal_sigma'] = 0.0
atoms.arrays['thermal_sigma'] = np.zeros(len(atoms))

gpts, sampling = grid_from_pixel_size(atoms, pixel_size_y, pixel_size_x)
ny, nx = gpts

cell_thickness = atoms.get_cell()[2, 2]
slice_dz = cell_thickness / n_slices_per_cell

wk_param = WeickenmeierKohlParametrization()

# WK potential for KG methods
potential_wk = abtem.Potential(
    atoms, gpts=gpts, slice_thickness=slice_dz,
    projection="finite", parametrization=wk_param,
)
pot_array_wk = jnp.array(cupy.asnumpy(potential_wk.build(lazy=False).array / slice_dz))

# Lobato potential for MS/WPM methods
potential_lobato = abtem.Potential(
    atoms, gpts=gpts, slice_thickness=slice_dz,
    projection="finite", parametrization="lobato",
)
pot_array_lobato = jnp.array(cupy.asnumpy(potential_lobato.build(lazy=False).array / slice_dz))

# Plane wave and sampling
pw = abtem.PlaneWave(energy=energy)
pw.grid.match(potential_wk)
psi0 = jnp.array(cupy.asnumpy(pw.build(lazy=False).array))
sampling = (float(pw.grid.sampling[0]), float(pw.grid.sampling[1]))
wavelength = energy2wavelength(energy)
beam_target = beam_for_angle_mrad(gpts, sampling, wavelength, beam_target_mrad, axis=beam_target_axis)

tracked_beams = [("00", 0, 0, "Pixel [0, 0]")]
if beam_target["visible"]:
    tracked_beams.append((
        "target",
        beam_target["h"],
        beam_target["k"],
        f"Pixel [{beam_target['h']}, {beam_target['k']}] ({beam_target['actual_mrad']:.1f} mrad)",
    ))

print(f"Au FCC, a = {a_central} Å, {n_slices_per_cell} slices/cell")
print(f"dz = {slice_dz:.4f} Å, energy = {energy/1e3:.0f} keV, λ = {wavelength:.4f} Å")
print(
    f"Grid: {ny}×{nx}, requested pixel size (y, x) = "
    f"({pixel_size_y:.4f}, {pixel_size_x:.4f}) Å"
)
print(f"Actual sampling (y, x) = ({sampling[0]:.4f}, {sampling[1]:.4f}) Å")
if beam_target["visible"]:
    print(
        f"{beam_target_mrad:.1f} mrad beam is visible on {beam_target_axis}: "
        f"using pixel [{beam_target['h']}, {beam_target['k']}] "
        f"({beam_target['actual_mrad']:.2f} mrad)."
    )
else:
    print(
        f"{beam_target_mrad:.1f} mrad beam is not visible on {beam_target_axis} "
        f"with this sampling; maximum visible angle is "
        f"{beam_target['max_visible_mrad']:.2f} mrad. Skipping that beam."
    )


## Thickness sweep — Fresnel MS, Angular Spectrum MS, WPM

Run the three paraxial/semi-paraxial methods through 0–25 unit cells using Lobato parametrization.

In [ ]:
fk = jnp.array(fresnel_propagation_kernel(ny, nx, sampling, z=slice_dz, energy=energy))
ak = jnp.array(angular_spectrum_propagation_kernel(ny, nx, sampling, z=slice_dz, energy=energy))

methods = ("ms", "as", "wpm")
results = {f"{method}_{beam_key}": [] for method in methods for beam_key, *_ in tracked_beams}
w_ms, w_as, w_wpm = psi0, psi0, psi0

for i in tqdm(range(len(n_cells_range)), desc="MS/WPM sweep"):
    if i > 0:
        w_ms, _, _ = simulate_fresnel_as(pot_array_lobato, w_ms, fk, slice_dz, energy)
        w_as, _, _ = simulate_fresnel_as(pot_array_lobato, w_as, ak, slice_dz, energy)
        w_wpm, _, _ = simulate_wpm(pot_array_lobato, w_wpm, slice_dz, energy, sampling)
        w_ms = jnp.array(w_ms)
        w_as = jnp.array(w_as)
        w_wpm = jnp.array(w_wpm)

    waves = {"ms": w_ms, "as": w_as, "wpm": w_wpm}
    for method, wave in waves.items():
        wave_np = np.asarray(wave)
        for beam_key, h, k, _ in tracked_beams:
            results[f"{method}_{beam_key}"].append(beam_amplitude_normalized(wave_np, h, k))

beam_results = {k: np.array(v) for k, v in results.items()}
sweep_exit = {"ms": w_ms, "as": w_as, "wpm": w_wpm}
print(f"Sweep complete: {len(n_cells_range)} unit cells")

## Second-Order Wave ODE thickness sweep (second-order, WK parametrization)

In [ ]:
kg_ode_results = {beam_key: [] for beam_key, *_ in tracked_beams}
w_kg = psi0
phi_kg = None

for i in tqdm(range(len(n_cells_range)), desc="Full KG ODE sweep"):
    if i > 0:
        w_kg, phi_kg, _, _ = simulate_kg_ode_full(
            pot_array_wk, w_kg, slice_dz, energy, sampling,
            initial_phi=phi_kg,
        )
        w_kg = jnp.array(w_kg)

    w_kg_np = np.asarray(w_kg)
    for beam_key, h, k, _ in tracked_beams:
        kg_ode_results[beam_key].append(beam_amplitude_normalized(w_kg_np, h, k))

kg_ode_results = {key: np.array(values) for key, values in kg_ode_results.items()}
final_summary = ", ".join(
    f"{label} final: {kg_ode_results[beam_key][-1]:.6f}"
    for beam_key, _, _, label in tracked_beams
)
print(f"KG ODE {final_summary}")

## Comparison plots - beam amplitudes vs Full KG ODE reference

Method colors, line styles, and markers are matched to the CBED notebook for visual consistency across figures. The paper-reference curves are intentionally omitted from this plotting view.

This cell produces one combined publication diagram: the full 0-100 unit-cell range on the top row and the 0-25 unit-cell view on the bottom row.

In [ ]:
from matplotlib.ticker import MultipleLocator

x = np.array(list(n_cells_range), dtype=float)
full_mask = np.ones_like(x, dtype=bool)
first25_mask = x <= 25

cell_thickness_nm = float(cell_thickness) * 0.1

def unit_cells_to_nm(values):
    return np.asarray(values, dtype=float) * cell_thickness_nm

method_colors = {
    "Fresnel MS": "C0",
    "Angular Spectrum MS": "C1",
    "WPM": "C2",
    "Full KG ODE": "0.10",
}
method_line_styles = {
    "Fresnel MS": "-",
    "Angular Spectrum MS": "--",
    "WPM": "-.",
    "Full KG ODE": "-",
}
method_result_keys = {
    "Fresnel MS": "ms",
    "Angular Spectrum MS": "as",
    "WPM": "wpm",
}

methods_by_beam = {}
for beam_key, _, _, _ in tracked_beams:
    methods_by_beam[beam_key] = {
        name: beam_results[f"{result_key}_{beam_key}"]
        for name, result_key in method_result_keys.items()
    }
    methods_by_beam[beam_key]["Full KG ODE"] = kg_ode_results[beam_key]

ordered_labels = ["Fresnel MS", "Angular Spectrum MS", "WPM", "Full KG ODE"]
plot_order = ["Full KG ODE", "Fresnel MS", "Angular Spectrum MS", "WPM"]

n_beams_to_plot = len(tracked_beams)
fig, axes = plt.subplots(2, n_beams_to_plot, figsize=(6.4 * n_beams_to_plot, 10), sharey=False)
axes = np.asarray(axes).reshape(2, n_beams_to_plot)
legend_handles = {}

for col, (beam_key, _, _, beam_label) in enumerate(tracked_beams):
    for row, (x_mask, range_title) in enumerate(((first25_mask, "0-25 unit cells"), (full_mask, "0-100 unit cells"))):
        ax = axes[row, col]
        x_view = x[x_mask]
        x_view_nm = unit_cells_to_nm(x_view)
        for name in plot_order:
            y_view = np.asarray(methods_by_beam[beam_key][name])[x_mask]
            is_reference = name == "Full KG ODE"
            line, = ax.plot(
                x_view_nm,
                y_view,
                color=method_colors[name],
                linestyle=method_line_styles[name],
                linewidth=2.8 if is_reference else 1.9,
                alpha=0.78 if is_reference else 0.95,
                label=name,
                zorder=1 if is_reference else 3,
            )
            if name not in legend_handles:
                legend_handles[name] = line
        ax.set_title(f"{beam_label} ({range_title})", pad=8)
        ax.set_xlabel("Thickness (nm)")
        ax.grid(True, alpha=0.28)
        ax.set_xlim(x_view_nm.min(), x_view_nm.max())
        ax.xaxis.set_minor_locator(MultipleLocator(cell_thickness_nm))
        ax.tick_params(axis="x", which="minor", length=2.5)

# On-axis [0,0] beam — first two plots (top and bottom row of column 0)
axes[0, 0].set_ylim(0.7, 1.0)
axes[1, 0].set_ylim(0.7, 1.0)

# Off-axis beam(s) — column 1, different y-limit per row
if n_beams_to_plot >= 2:
    axes[0, 1].set_ylim(0.0, 0.030)
    axes[1, 1].set_ylim(0.0, 0.060)

axes[0, 0].set_ylabel("Normalized beam amplitude")
axes[1, 0].set_ylabel("Normalized beam amplitude")

fig.legend(
    [legend_handles[name] for name in ordered_labels],
    ordered_labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.965),
    ncol=len(ordered_labels),
    frameon=False,
    fontsize=9,
    handlelength=2.6,
 )
fig.suptitle(
    f"Au [100] beam-amplitude verification at {energy/1e3:.0f} keV; "
    f"{n_slices_per_cell} slices/cell; sampling=({sampling[0]:.4f}, {sampling[1]:.4f}) Å",
    fontsize=11,
    y=1.01,
 )
fig.tight_layout(rect=[0.0, 0.05, 1.0, 0.93])
plt.show()

print("RMSE computed over the full simulated thickness range against Full KG ODE")
header = f"{'Method':<25s}" + "".join(f" {label + ' RMSE':>24s}" for _, _, _, label in tracked_beams)
print(header)
print("-" * len(header))
for name in ordered_labels:
    if name == "Full KG ODE":
        continue
    row = f"{name:<25s}"
    for beam_key, _, _, _ in tracked_beams:
        rmse = curve_rmse(methods_by_beam[beam_key][name], methods_by_beam[beam_key]["Full KG ODE"])
        row += f" {rmse:>24.6f}"
    print(row)


In [ ]:
# Export combined figure for paper
if "fig" not in globals():
    raise RuntimeError("Run the comparison-plot cell first so the combined figure exists.")

paper_fig_dir = resolve_paper_figures_dir()
output_path = paper_fig_dir / "Au_pixel_amplitudes.pdf"
fig.savefig(output_path, format="pdf", bbox_inches="tight", dpi=300)
if not output_path.exists() or output_path.stat().st_size == 0:
    raise RuntimeError(f"Failed to write figure: {output_path}")

old_first25_path = paper_fig_dir / "Au_pixel_amplitudes_first25_unit_cells.pdf"
if old_first25_path.exists():
    old_first25_path.unlink()

print(f"Saved -> {output_path}")
if not old_first25_path.exists():
    print(f"Removed obsolete separate first-25 figure -> {old_first25_path}")


## Exit-wave and image-plane comparison at final thickness

The columns show exit-wave intensity and phase, followed by the intensity after a representative aberration-corrected 300 keV HRTEM objective-lens transfer function. A residual $C_s = 10$ µm and a 12 mrad objective aperture are used. The positive defocus is selected so the aberration phase increases monotonically from zero to $+\pi/2$ at the aperture edge; consequently, the phase CTF $\sin(\chi)$ remains positive throughout the transmitted band and does not invert contrast. A 2 nm focal-spread envelope is also applied. Display ranges are shared by column so differences between propagation methods remain visible.


In [ ]:
exit_wave_methods = [
    ("Full KG ODE", w_kg),
    ("WPM", sweep_exit["wpm"]),
    ("Angular Spectrum MS", sweep_exit["as"]),
    ("Fresnel MS", sweep_exit["ms"]),
]


# Representative aberration-corrected 300 keV HRTEM settings. Choose the
# defocus so sin(chi) rises from 0 to +1 without a contrast reversal.
microscope_Cs_um = 10.0
microscope_Cs_A = microscope_Cs_um * 1e4
microscope_aperture_mrad = 12.0
microscope_aperture_rad = microscope_aperture_mrad * 1e-3
microscope_target_phase_rad = 0.5 * np.pi
microscope_defocus_A = (
    wavelength * microscope_target_phase_rad
    / (np.pi * microscope_aperture_rad**2)
    - 0.5 * microscope_Cs_A * microscope_aperture_rad**2
)
microscope_defocus_nm = microscope_defocus_A * 0.1
microscope_focal_spread_nm = 2.0

reference_shape = np.asarray(exit_wave_methods[0][1]).shape
objective_transfer = objective_lens_transfer_function(
    reference_shape,
    sampling,
    wavelength,
    microscope_defocus_A,
    microscope_Cs_A,
    microscope_aperture_mrad,
    microscope_focal_spread_nm,
)

exit_wave_intensities = {}
exit_wave_phases = {}
image_plane_intensities = {}

for name, wave in exit_wave_methods:
    wave = np.asarray(wave)
    exit_wave_intensities[name] = np.abs(wave) ** 2
    exit_wave_phases[name] = np.angle(global_phase_removed(wave))

    image_wave = np.fft.ifft2(np.fft.fft2(wave) * objective_transfer)
    image_plane_intensities[name] = np.abs(image_wave) ** 2

reference_image_intensity = image_plane_intensities["Full KG ODE"]
reference_image_rms = np.sqrt(np.mean(reference_image_intensity**2))
image_plane_nrmse = {
    name: np.sqrt(np.mean((values - reference_image_intensity) ** 2))
    / reference_image_rms
    for name, values in image_plane_intensities.items()
}

intensity_limits = shared_percentile_limits(exit_wave_intensities.values())
image_intensity_limits = shared_percentile_limits(image_plane_intensities.values())

extent_nm = [
    0.0,
    reference_shape[1] * sampling[1] * 0.1,
    reference_shape[0] * sampling[0] * 0.1,
    0.0,
]

n_methods = len(exit_wave_methods)
n_columns = 3
exit_fig = plt.figure(figsize=(11.8, 11.6), constrained_layout=True)
exit_grid = exit_fig.add_gridspec(
    n_methods,
    2 * n_columns,
    width_ratios=[1.0, 0.045, 1.0, 0.045, 1.0, 0.045],
    wspace=0.04,
)

exit_axes = np.empty((n_methods, n_columns), dtype=object)
reference_axis = None
for row in range(n_methods):
    for column in range(n_columns):
        axis_kwargs = {}
        if reference_axis is not None:
            axis_kwargs = {"sharex": reference_axis, "sharey": reference_axis}
        exit_axes[row, column] = exit_fig.add_subplot(
            exit_grid[row, 2 * column],
            **axis_kwargs,
        )
        if reference_axis is None:
            reference_axis = exit_axes[row, column]

colorbar_axes = [
    exit_fig.add_subplot(exit_grid[:, 2 * column + 1])
    for column in range(n_columns)
]

column_titles = [
    "Exit-wave intensity",
    "Exit-wave phase\n(global phase removed)",
    f"TEM image intensity\n(non-inverting defocus = {microscope_defocus_nm:+.1f} nm)",
]
for ax, title in zip(exit_axes[0], column_titles):
    ax.set_title(title, fontsize=10)

image_handles = [None] * n_columns
for row, (name, _) in enumerate(exit_wave_methods):
    values_by_column = [
        exit_wave_intensities[name],
        exit_wave_phases[name],
        image_plane_intensities[name],
    ]
    plot_options = [
        dict(cmap="gray", vmin=intensity_limits[0], vmax=intensity_limits[1]),
        dict(cmap="twilight", vmin=-np.pi, vmax=np.pi),
        dict(
            cmap="gray",
            vmin=image_intensity_limits[0],
            vmax=image_intensity_limits[1],
        ),
    ]

    for column, (ax, values, options) in enumerate(
        zip(exit_axes[row], values_by_column, plot_options)
    ):
        image_handles[column] = ax.imshow(values, extent=extent_nm, **options)
        ax.tick_params(labelsize=8)
        if row != n_methods - 1:
            ax.tick_params(labelbottom=False)
        else:
            ax.set_xlabel("x (nm)", fontsize=9)
        if column != 0:
            ax.tick_params(labelleft=False)

    exit_axes[row, 0].set_ylabel(f"{name}\ny (nm)", fontsize=9)
    if name == "Full KG ODE":
        comparison_text = "reference"
    else:
        comparison_text = f"NRMSE = {image_plane_nrmse[name]:.2e}"
    exit_axes[row, 2].text(
        0.97,
        0.04,
        comparison_text,
        transform=exit_axes[row, 2].transAxes,
        ha="right",
        va="bottom",
        fontsize=7.5,
        color="white",
        bbox={"facecolor": "black", "alpha": 0.55, "edgecolor": "none", "pad": 1.5},
    )

colorbar_labels = ["Intensity", "Phase (rad)", "Intensity"]
for image_handle, colorbar_axis, label in zip(
    image_handles,
    colorbar_axes,
    colorbar_labels,
):
    colorbar = exit_fig.colorbar(image_handle, cax=colorbar_axis)
    colorbar.set_label(label, fontsize=8)
    colorbar.ax.tick_params(labelsize=7)

exit_fig.suptitle(
    f"Au [100] exit wave and HRTEM image after {x[-1]:.0f} unit cells "
    f"({unit_cells_to_nm([x[-1]])[0]:.2f} nm) at {energy/1e3:.0f} keV\n"
    f"$C_s$ = {microscope_Cs_um:.0f} $\\mu$m, defocus = "
    f"{microscope_defocus_nm:+.1f} nm, aperture = {microscope_aperture_mrad:.0f} mrad, "
    f"focal spread = {microscope_focal_spread_nm:.0f} nm",
    fontsize=11,
)
plt.show()

print("Representative aberration-corrected HRTEM transfer settings")
print(f"Residual Cs: {microscope_Cs_um:.1f} um")
print(f"Non-inverting defocus: {microscope_defocus_nm:+.2f} nm")
print(f"Objective aperture semi-angle: {microscope_aperture_mrad:.1f} mrad")
print(f"Phase at aperture edge: {microscope_target_phase_rad / np.pi:.2f} pi")
print(f"Focal spread: {microscope_focal_spread_nm:.1f} nm")
print("\nImage-plane intensity NRMSE against Full KG ODE")
for name, _ in exit_wave_methods:
    if name != "Full KG ODE":
        print(f"{name:<25s} {image_plane_nrmse[name]:.6e}")


In [ ]:
# Export exit-wave comparison figure for paper
if "exit_fig" not in globals():
    raise RuntimeError("Run the exit-wave comparison cell first so the figure exists.")

paper_fig_dir = resolve_paper_figures_dir()
exit_output_path = paper_fig_dir / "Au_exit_wave_amplitudes.pdf"
exit_fig.savefig(exit_output_path, format="pdf", bbox_inches="tight", dpi=300)
if not exit_output_path.exists() or exit_output_path.stat().st_size == 0:
    raise RuntimeError(f"Failed to write figure: {exit_output_path}")

print(f"Saved -> {exit_output_path}")


## RMSE summary vs Full KG ODE

In [ ]:
print("% Paste into Paper/main.tex  ->  tab:rmse_verification")
print(r"\begin{tabular}{l" + "r" * len(tracked_beams) + "}")
print(r"    \toprule")
beam_headers = " & ".join(
    f"\\textbf{{RMSE ${label}$ vs KG ODE}}"
    for _, _, _, label in tracked_beams
)
print(r"    \textbf{Method} & " + beam_headers + r" \\")
print(r"    \midrule")
latex_names = {
    "Fresnel MS": r"Fresnel multislice",
    "Angular Spectrum MS": r"Angular-spectrum multislice",
    "WPM": r"WPM",
}
for name in ["Fresnel MS", "Angular Spectrum MS", "WPM"]:
    values = []
    for beam_key, *_ in tracked_beams:
        rmse = curve_rmse(methods_by_beam[beam_key][name], methods_by_beam[beam_key]["Full KG ODE"])
        values.append(f"{rmse:.4e}")
    print(f"    {latex_names[name]:<35s} & " + " & ".join(values) + r" \\")
print(r"    \bottomrule")
print(r"\end{tabular}")